# 🧪 Test AWS Bedrock + Qdrant Configuration

This notebook tests your Bedrock and Qdrant setup.

## Prerequisites

Make sure you have:
1. ✅ AWS credentials in `.env` file
2. ✅ Bedrock models enabled in AWS Console
3. ✅ Qdrant credentials configured (already done!)
4. ✅ Installed dependencies: `pip install -r requirements.txt`

## Step 1: Load Environment

In [ ]:
import sys
from pathlib import Path

# Add project root to path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from utils.env_loader import load_environment, check_required_keys, display_config

# Load environment
load_environment()

# Check required keys
check_required_keys(
    'AWS_ACCESS_KEY_ID',
    'AWS_SECRET_ACCESS_KEY',
    'QDRANT_API_KEY'
)

# Display configuration
display_config()

## Step 2: Test Complete Stack

In [ ]:
from utils.bedrock_helper import test_full_stack

# Test everything
results = test_full_stack()

# Show results
print("\n" + "="*60)
print("Test Results Summary:")
print("="*60)
print(f"Bedrock Status: {results['bedrock']['status']}")
print(f"Qdrant Status: {results['qdrant']['status']}")
print(f"Overall Status: {results['overall_status']}")
print("="*60)

## Step 3: Test Individual Components

### Test Bedrock Embeddings

In [ ]:
from utils.bedrock_helper import get_bedrock_embeddings

# Get embeddings model
embeddings = get_bedrock_embeddings()

# Test with a sample text
test_text = "AWS Bedrock is a fully managed service for building AI applications."
embedding_vector = embeddings.embed_query(test_text)

print(f"✅ Embeddings Model: {embeddings.model_id}")
print(f"✅ Vector Dimension: {len(embedding_vector)}")
print(f"✅ First 5 values: {embedding_vector[:5]}")

### Test Bedrock LLM

In [ ]:
from utils.bedrock_helper import get_bedrock_llm

# Get LLM
llm = get_bedrock_llm()

# Test with a simple query
response = llm.invoke("What is RAG in AI? Answer in one sentence.")

print(f"✅ LLM Model: {llm.model_id}")
print(f"✅ Response: {response.content}")

### Test Qdrant Connection

In [ ]:
from utils.bedrock_helper import get_qdrant_client

# Get Qdrant client
client = get_qdrant_client()

# Get collections
collections = client.get_collections()

print(f"✅ Qdrant connected successfully!")
print(f"✅ Number of collections: {len(collections.collections)}")
print("\nExisting collections:")
for col in collections.collections:
    print(f"  - {col.name}: {col.vectors_count} vectors")

## Step 4: Complete RAG Pipeline Test

Let's test a complete RAG pipeline with sample documents.

In [ ]:
from utils.bedrock_helper import quick_setup
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Quick setup
llm, embeddings, vectorstore = quick_setup("test_rag_collection")

print("\n✅ RAG pipeline initialized!")

### Add Sample Documents

In [ ]:
# Sample documents about RAG
sample_docs = [
    Document(
        page_content="""
        Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval 
        with text generation. It retrieves relevant documents from a knowledge base and uses 
        them to generate more accurate and informed responses.
        """,
        metadata={"source": "rag_intro", "topic": "basics"}
    ),
    Document(
        page_content="""
        AWS Bedrock provides access to foundation models from leading AI companies through a 
        single API. It includes models like Claude 3 from Anthropic, which excels at reasoning 
        and analysis tasks, making it ideal for RAG applications.
        """,
        metadata={"source": "bedrock_info", "topic": "aws"}
    ),
    Document(
        page_content="""
        Qdrant is a vector database optimized for similarity search and RAG applications. 
        It supports filtering, payload storage, and efficient nearest neighbor search, 
        making it perfect for production RAG systems.
        """,
        metadata={"source": "qdrant_info", "topic": "vector_db"}
    ),
    Document(
        page_content="""
        Embeddings are dense vector representations of text that capture semantic meaning. 
        Amazon Titan Embeddings V2 produces 1024-dimensional vectors that are highly effective 
        for semantic search and retrieval tasks.
        """,
        metadata={"source": "embeddings_info", "topic": "embeddings"}
    ),
]

# Add documents to Qdrant
print("Adding documents to Qdrant...")
vectorstore.add_documents(sample_docs)
print(f"✅ Added {len(sample_docs)} documents to vector store")

### Test Retrieval

In [ ]:
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Test query
query = "What is RAG and how does it work?"
print(f"Query: {query}\n")

# Retrieve relevant documents
relevant_docs = retriever.get_relevant_documents(query)

print(f"✅ Retrieved {len(relevant_docs)} relevant documents:\n")
for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:")
    print(f"Content: {doc.page_content[:150]}...")
    print(f"Metadata: {doc.metadata}")
    print()

### Test Complete RAG Chain

In [ ]:
from langchain.chains import RetrievalQA

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Ask a question
question = "What is RAG and why is it useful?"
print(f"Question: {question}\n")

result = qa_chain({"query": question})

print("Answer from Claude 3 (via Bedrock):")
print("="*60)
print(result["result"])
print("="*60)

print(f"\n✅ Used {len(result['source_documents'])} source documents from Qdrant")

### Test Another Query

In [ ]:
# Test with a different question
question2 = "What are embeddings and how do they help in RAG?"
print(f"Question: {question2}\n")

result2 = qa_chain({"query": question2})

print("Answer:")
print("="*60)
print(result2["result"])
print("="*60)

## Step 5: Verify Collection in Qdrant

In [ ]:
from utils.bedrock_helper import get_qdrant_client

client = get_qdrant_client()

# Get collection info
collection_info = client.get_collection("test_rag_collection")

print("Collection Information:")
print("="*60)
print(f"Name: {collection_info.config.params.vectors.size}")
print(f"Vector Size: {collection_info.config.params.vectors.size}")
print(f"Distance Metric: {collection_info.config.params.vectors.distance}")
print(f"Number of Vectors: {collection_info.points_count}")
print("="*60)

## ✅ Success!

If all cells ran successfully, your Bedrock + Qdrant setup is working perfectly!

### What We Tested:

1. ✅ Environment configuration loaded
2. ✅ AWS Bedrock connection
3. ✅ Claude 3 Sonnet LLM
4. ✅ Titan Embeddings V2
5. ✅ Qdrant Cloud connection
6. ✅ Document embedding and storage
7. ✅ Semantic search/retrieval
8. ✅ Complete RAG pipeline with Q&A

### Next Steps:

1. **Explore notebooks**: Try `notebooks/simple_rag/simple_rag.ipynb`
2. **Advanced techniques**: Graph RAG, Agentic RAG
3. **Production tips**: See `BEDROCK_QDRANT_SETUP.md`

### Your Stack:

- **LLM**: Claude 3 Sonnet (via AWS Bedrock)
- **Embeddings**: Titan Embeddings V2 (1024 dims)
- **Vector DB**: Qdrant Cloud (EU West 2)
- **Status**: 🟢 Fully Operational

**Happy RAG building!** 🚀